<a href="https://colab.research.google.com/github/ffserro/analise-de-dados-boas-praticas/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Introdução

## 1.1. Contexto do problema

## 1.2. Objetivo geral

## 1.3. Objetivos específicos

## 1.4. Breve configuração do ambiente

Antes de começar, peço que aguardem a instalação das dependências do projeto.

Após a instalação, será necessário reiniciar o embiente de execução.

Pede-se apenas que se clique novamente em "Executar tudo" após o término da instalação.

In [ ]:
%%capture

!wget -nc 'https://github.com/ffserro/analise-de-dados-boas-praticas/raw/refs/heads/main/pyproject.toml'
!pip install -e .

In [ ]:
from warnings import filterwarnings
filterwarnings('ignore')

# 2. Base de Dados

A base de dados será carregada com a biblioteca Polars.

O motivo dessa escolha foi que esta biblioteca seria mais rápida do que o Pandas quando tratamos de DataFrames com mais de 1e6 linhas. Acontece que durante a limpeza o conjunto dos dados acabou ficando menor do que isso, mas como eu gostei de aprender sobre a biblioteca e achei a escrita bem mais elegante e mais fácil de vetorizar grandes transformações em comparação com o Pandas, decidi manter o uso predominante do Polars.

In [ ]:
import polars as pl

df = (
    pl.read_parquet([f'https://github.com/ffserro/analise-de-dados-boas-praticas/raw/refs/heads/main/database_2/metadata_part_{i:04d}.parquet' for i in range(1, 6)])
    .join(
        pl.read_parquet([f'https://github.com/ffserro/analise-de-dados-boas-praticas/raw/refs/heads/main/database_2/textos_part_{i:04d}.parquet' for i in range(1, 24)]),
        on='id'
    )
    .with_columns(
        pl.col('pubDate').str.to_date(),
        pl.col('pubName').str.head(3),  # Já com uma primeira limpeza porque existem algumas edições especiais que  tem um 'E' no final, como DO1E, por exemplo
    )
    .filter(
        pl.col('artCategory').str.to_lowercase().str.starts_with('ministério da defesa'),
        pl.col('artCategory').str.to_lowercase().str.contains(r'marinha|ex[ée]rcito|aeron[áa]utica')    # Como a proposta é fazer uma análise entre as três forças, não nos interessam os órgãos que não sejam subordinados ao Ministério da Defesa, nem as publicações do próprio Ministério.
    )
)

df.tail()

In [ ]:
df.shape

## 2.1. Origem dos dados

Todos os dados utilizados foram extraídos da [Base de Dados de Publicações do DOU](https://in.gov.br/acesso-a-informacao/dados-abertos/base-de-dados).

O dicionário de dados ainda não foi disponibilizado segundo o próprio site, porém há uma breve explicação sobre a divisão dos documentos entre Seções.

Os documentos referentes às publicações estão no formato .xml, compactados em um arquivo .zip por ano, mês e seção.

## 2.2. Etapas de coleta

Toda a coleta foi realizada com um crawler básico, que está no repositório neste [link](https://github.com/ffserro/analise-de-dados-boas-praticas/blob/main/data_fetcher/pack_downloader.py).

Ele busca os links no site dos dados abertos do DOU, enviando um request com ano e mês. Precisa ser feito dessa forma, porque os links são gerados por seção, e se não contiverem informações dos cookies, tentar baixar os arquivos retorna um bad request.

Depois de resgatados os links, imediatamente se inicia o download do arquivo .zip em bytes para uma pasta local, onde é descompactado.

Após descompactar, eu fiz uma primeira filtragem para deletar todos os arquivos que não possuíssem em seu texto o termo "Ministério da Defesa".

A pasta com os arquivos baixados não pode ser armazenada no repositório porque possuía uma grande quantidade de arquivos, além do limite permitido pelo github. A solução foi já construir um dataset preliminar, no formato .parquet por ser mais leve, e salvá-lo particionado para que o git lfs permitisse o upload para o repositório. Essa etapa foi realizada por este [script](https://github.com/ffserro/analise-de-dados-boas-praticas/blob/main/bd_utils/bd_builder.py).

Durante a etapa de transformar os arquivos .xml em um dataframe, foram extraídas as informações tag por tag usando regex puro. Assim é a estrutura padrão de um arquivo:
```
<xml>
    <article id="" name="" idOficio="" pubName="" artType="" pubDate="" artClass="" artCategory="" artSize="" artNotes="" numberPage="" pdfPage="" editionNumber="" highlightType="" highlightPriority="" highlight="" highlightimage="" highlightimagename="" idMateria="">
        <body>
            <Identifica> </Identifica>
            <Data> </Data>
            <Ementa> </Ementa>
            <Titulo> </Titulo>
            <SubTitulo> </SubTitulo>
            <Texto> </Texto>
            <Autores>
                <assina> </assina>
                <cargo> </cargo>
            </Autores>
        </body>
        <Midias> </Midias>
    </article>
</xml>
```
Desta estrutura foram extraídos tanto os atributos da tag `article` quanto o conteúdo textual das tags `Identifica`, `Data`, `Ementa`, `Titulo`, `SubTitulo`, `Texto`, `Autores` e transformados em colunas deste dataframe.

## 2.3. Período analisado

São disponibilizados pela Base de Dados Abertos do DOU as publicações desde o ano de 2002 até o ano de 2026.

In [ ]:
from plotly import express as px
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt

volume_por_ano = (
    df
    .group_by(pl.col('pubDate').dt.strftime('%Y/%m'), pl.col('pubName').str.head(3))
    .agg(pl.len())
    .sort('pubDate', 'pubName')
)

fig = px.area(
    volume_por_ano.to_pandas(),
    x='pubDate',
    y='len',
    color='pubName',
    title='Volume de artigos por mês e seção',
    labels={'len':'Volume', 'pubDate':'Mês de referência', 'pubName':'Seção'},
    render_mode='webgl'
)
fig.show()

# plt.figure(figsize=(20,8))

# (
#     so.Plot(volume_por_ano.to_pandas(), 'pubDate', 'len', color='pubName').add(so.Area(alpha=.7), so.Stack())
# )

Pode ser observado uma quantidade de dados faltosos durante todo o ano de 2017.

Provavelmente a própria plataforma do DOU estava sendo reformulada durante esse período, porque se observam dados muito mais estruturados e consistentes a partir do ano de 2018.

Como exemplo mostro abaixo uma das colunas do dataframe que exemplifica essa melhor estruturação:

In [ ]:
pl.concat(
    [
        df.filter(pl.col('pubDate').dt.year() < 2018).select(pl.col('artType').unique().alias('type_antes')),
        df.filter(pl.col('pubDate').dt.year() >= 2018).select(pl.col('artType').unique().alias('type_depois')),
    ],
    how='horizontal'
).drop_nulls().show(limit=20, fmt_str_lengths=50)

Desta forma, por simplicidade e para limitar o escopo, vou considerar apenas os dados a partir do ano de 2018.

In [ ]:
df = (
    df
    .filter(pl.col('pubDate').dt.year() >= 2018)
)

(
    df
    .select(
        pl.col('pubDate').min().alias('de'),
        pl.col('pubDate').max().alias('ate')
    )
)

## 2.4. Limpeza e limitações

### 2.4.1. Valores nulos

Inicio a limpeza dos dados verificando a quantidade de valores nulos que cada uma das colunas contém, expressa aqui como razão entre a quantidade de nulos e o total de observações no conjunto de dados.

In [ ]:
df.select(
    (pl.all().null_count() / pl.len() * 100).round(2).name.suffix("_null_pct")
).transpose(include_header=True).sort(by='column_0', descending=True).show(limit=15)

As colunas `titulo`, `autores`, `artSection`, `subtitulo`, `ementa` e `data`, embora pudessem conter dados muito úteis para análise, possuem uma quantidade insignificante de dados presentes, não podendo então serem consideradas. Removerei estas colunas do dataframe.

In [ ]:
df = df.drop('titulo', 'autores', 'artSection', 'subtitulo', 'ementa', 'data')

### 2.4.2. Valores únicos

A próxima verificação é a quantidade de valores únicos por coluna. Algumas colunas naturalmente são compostas na sua totalidade por valores nulos, como é o exemplo do `id`, que identifica uma observação. No entanto, para uma análise dos dados, com exceção da coluna que seria a chave primária (`id`), ter outras colunas com dados únicos não permite extrair muita informação, já que não fornece um padrão de semelhança ou agrupamento das observações.

In [ ]:
df.select(
    (pl.all().n_unique() / pl.len() * 100).round(2).name.suffix('_unique_pct')
).transpose(include_header=True).sort(by='column_0', descending=True).show(limit=15)

No primeiro momento notei que `idMateria`, que me parece que deveria ter 100% de valores únicos, se repete algumas vezes. Por quê?

In [ ]:
df.filter(pl.col('idMateria').is_duplicated() & ~pl.col('idMateria').is_null()).sort(by=['idMateria', 'numberPage'], descending=[True, False]).select('idMateria', 'numberPage', 'texto')

Algumas matérias, por serem mais extensas, tiveram seu texto ocupando mais do que uma página da publicação do Diário Oficial daquela data, por isso o texto aparece partido.

Abaixo, eu restituo os fragmentos dos textos ao seu preâmbulo, unificando sob uma observação. Espera-se com isso que a coluna `idMateria` se torne uma coluna de valores únicos.

In [ ]:
df = (
    df
    .select(pl.all().exclude('texto'))
    .sort(by=['idMateria', 'numberPage'], descending=[True, False])
    .unique('idMateria', keep='first')
    .join(
        (
            df
            .sort(by=['idMateria', 'numberPage'], descending=[True, False])
            .group_by('idMateria')
            .agg(pl.col('texto').str.join('\n'))
        ), on='idMateria'
    )
)

Primeiro foi criado um dataframe sem a coluna `texto` ao qual foram reintegrados os textos devidamente concatenados.

In [ ]:
df.select(
    (pl.all().n_unique() / pl.len() * 100).round(2).name.suffix('_unique_pct')
).transpose(include_header=True).sort(by='column_0', descending=True).show(limit=15)

Assim, os identificadores de matéria se tornaram únicos, em mesma quantidade que os identificadores dos documentos.

In [ ]:
(
    df
    .select(
        pl.col('idMateria').n_unique().alias('contagem_materias'),
        pl.col('id').n_unique().alias('contagem_documentos')
    )
)

Não havendo mais necessidade de possuir dois identificadores diferentes, e como conhecer o número da página que contém a publicação não é mais útil, as duas colunas serão removidas.

In [ ]:
df = (
    df.drop('idMateria', 'numberPage')
)

Agora eu quero saber porque é que os nomes dos documentos e os identificadores dos ofícios não são únicos.

In [ ]:
df.sort(by=['pubDate', 'name']).filter(pl.col('name').is_duplicated()).select('name').show(limit=15, fmt_str_lengths=500)

Os nomes dos documentos não são padronizados e parece que eles contém as mesmas informações que a coluna `identifica`. Vejamos se temos mais sorte com a coluna `idOficio`.

In [ ]:
df.sort(by=['idOficio', 'pubDate']).filter(pl.col('idOficio').is_duplicated()).select('pubDate', 'idOficio').unique().show(limit=15)

Parece que todos os valores de `idOficio` duplicados possuem uma única `pubDate` comum. Vou verificar se isso está relacionado com o nome.

In [ ]:
df.filter(pl.col('idOficio')=='4704719').select('idOficio', 'name', 'texto').show(fmt_str_lengths=500)

É isso.

Os documentos são enviados pelas organizações em lotes para serem publicados. Estes documentos devem ser encaminhados por um Ofício, que agrega um ou mais documentos que são publicados no mesmo dia. Entendi o motivo de não possuir apenas dados únicos, mas o `idOficio` ainda não me permite vislumbrar uma aplicação direta dos seus dados. Por isso será retirado do dataframe.

In [ ]:
df = df.drop('idOficio')

Agora vamos descobrir por que a coluna `identifica`, que também soa como uma variável de identidade, não possui apenas valores únicos.

In [ ]:
df.sort(by=['pubDate', 'identifica']).filter(pl.col('identifica').is_duplicated()).select('identifica').unique().show(limit=15, fmt_str_lengths=500)

A primeira vista é ruim. Os valores de `identifica` não são padronizados. Possuem alguns valores genéricos, e outros valores específicos que não deveriam ser duplicados, como o mesmo número de contrato do mesmo ano para a mesma uasg.

In [ ]:
df.filter(pl.col('identifica')=='AVISO DE LICITAÇÃO').select('identifica', 'name', 'texto').show(fmt_str_lengths=500)

In [ ]:
df.filter(pl.col('identifica')=="EXTRATO DE TERMO ADITIVO Nº 1/2025 - UASG 168006").select('identifica', 'name', 'texto').show(fmt_str_lengths=500)

Os valores genéricos de `identifica` fornecem o mesmo dado que a coluna `artType`, e os valores específicos acabam por repetir o cabeçalho do `texto` do documento. Por isso vou remover a coluna `identifica`, e aproveitando, excluo também a coluna `name` porque seus dados também constam no mesmo cabeçalho.

In [ ]:
df =(
    df.drop('name', 'identifica')
)

Vamos descobrir porque a coluna `highlight`, que é um destaque do texto, possui tão poucos valores únicos.

In [ ]:
df.group_by('highlight').agg(pl.len()).sort('len', descending=True)

In [ ]:
df.filter(pl.col('highlight')!='')

A coluna `highlight` possui tão poucos valores únicos porque na verdade, o valor mais repetido é uma string vazia. Ou seja, são uma grande quantidade de valores ausentes.

Vejamos se o mesmo vale para toda a família das variáveis `highlight`.

In [ ]:
df.filter(pl.col('highlightType')!='')

Aqui nos despedimos das variáveis de `highlight`.

In [ ]:
df = (
    df.drop('highlight', 'highlightType', 'highlightPriority', 'highlightimage', 'highlightimagename')
)

In [ ]:
df.head()

Conheçamos a coluna `artNotes`.

In [ ]:
df.filter(pl.col('artNotes')!='').select('artNotes').unique().show(limit=15, fmt_str_lengths=500)

O motivo para remoção desta variável dispensa explicações. Talvez a sua existência é que exija informações.

Acredito que ele atenda necessidades muito internas dos órgãos ou da própria Imprensa Nacional.

In [ ]:
df = (
    df.drop('artNotes')
)

In [ ]:
df.head()

O que é a coluna `artSize` ?

In [ ]:
df.select('artSize').unique()

O nome sugere que existam 4 formatos de tamanhos diferentes que podem ser alocados para determinada publicação nas nas edições do DOU. Vejamos se isso faz sentido.

In [ ]:
(
    df
    .group_by('pubName', 'artSize')
    .agg(pl.len().alias('contagem'))
    .with_columns(
        (
            pl.col('contagem') / pl.col('contagem').sum().over('pubName') * 100
        ).round(2).alias('porcentagem')
    )
    .sort(by=['pubName', pl.col('artSize').str.to_integer()])
    .show(limit=-1)
)

É interessante saber que a maior parte dos documentos ocupa o tamanho padrão de 12, mas não vamos utilizar esses dados na nossa análise.

In [ ]:
df = (
    df.drop('artSize')
)

Vamos inspecionar agora a coluna `editionNumber`

In [ ]:
df.select('editionNumber').describe()

Desconfio de que o `editionNumber` seja uma contagem incremental das publicações diárias, ano a ano.

In [ ]:
df.select('pubDate', 'editionNumber').unique().sort(by='pubDate').show(limit=15)

In [ ]:
(
    df
    .group_by(pl.col('pubDate').dt.year().alias('ano'))
    .agg([
        pl.col('pubDate').n_unique().alias('contagem_dias'),
        pl.col('editionNumber').n_unique().alias('contagem_edicoes')
    ])
    .sort(by='ano')
)

É isso. O número de edições se aproxima muito do número de dias. A contage não é exatamente igual porque por vezes é publicada uma edição especial, sendo rotulada de DO1E, DO2E ou DO3E, mas eu já fiz essa limpeza no carregamento dos dados, por isso não ficou explícito aqui.

Como o `editionNumber` não agrega mais informação do que o `pubDate`, será removido do dataframe.

In [ ]:
df = df.drop('editionNumber')

A variável `pdfPage` é uma URL que redireciona para a publicação no formato .pdf, exatamente como foi publicada. Removerei do dataframe.

In [ ]:
df = df.drop('pdfPage')

Abaixo, só para reordenar as colunas que restaram.

In [ ]:
df = df.select('id', 'pubName', 'pubDate', 'artCategory', 'artType', 'artClass', 'texto')

In [ ]:
df.head()

Essa variável `artClass` parece classificar os documentos por alguma lógica. Estou interessado nela porque pode facilitar o processo de segmentar os documentos por assunto.

In [ ]:
(
    df
    .group_by('artClass', 'pubName', 'artType')
    .agg(pl.len())
    .sort(by=[pl.col('len').sum().over('artClass'), 'artClass'], descending=[True, False])
).show(limit=20, fmt_str_lengths=100)

Talvez, olhando rápido, parece que o número na penúltima posição quando é 0017 se trata de uma Portaria, quando é 0079 é um Aviso, quando é 0032 é um Extrato. Mas nem sempre. Vou tentar achar algum sentido nisso.

Vou assumir que cada um dos valores separados por : no `artClass` seriam coordenadas em um espaço vetorial com 12 dimensões.

In [ ]:
df_vetores = df.select(
    'id',
    (
        pl.col('artClass')
        .str.split(':')
        .list.eval(
            pl.element()
            .str.strip_chars('a')
            .cast(pl.Float64) * 1e-3
        )
    )
)

df_vetores

Vou explodir os valores dos caminhos hierárquicos do `artCategory` também.

A maioria segue até 5 níveis, partindo do Ministério da Defesa. Note que não existem 8, nem 9.

In [ ]:
df.select(pl.col('artCategory').str.split('/').list.len()).group_by('artCategory').agg(pl.len()).sort('artCategory')

Vou forçar todas essas variáveis suspeitas de possuírem alguma correlação para visualizar se alguma faz sentido. Abaixo estou improvisando dummies para as variáveis categóricas que eu vou tentar relacionar com o "vetor" do `artClass`.

In [ ]:
artCategory_dummy = {v:k for k,v in enumerate(df.select('artCategory').unique().sort('artCategory').to_series().to_list())}
artType_dummy = {v:k for k,v in enumerate(df.select('artType').unique().sort('artType').to_series().to_list())}
preType_dummy = {v:k for k,v in enumerate(df.select(pl.col('artType').str.split(' ').list.first()).unique().sort('artType').to_series().to_list())}

In [ ]:
import polars.selectors as cs
from seaborn import heatmap

heatmap((
    df_vetores
    .with_columns(
        pl.col('artClass').list.to_struct(
            fields=[f'vector_{i}' for i in range(12)]
        )
    )
    .unnest('artClass')
    .join(
        df,
        on='id'
    )
    .with_columns(
        forca=pl.col('artCategory').str.split('/').list.slice(1,1).list.first().map_elements(lambda x: {'Comando da Aeronáutica':0, 'Comando da Marinha':1, 'Comando do Exército':2}[x], return_dtype=pl.Int32).cast(pl.Float64),
        artCategory=pl.col('artCategory').map_elements(lambda x: artCategory_dummy[x], return_dtype=pl.Int32).cast(pl.Float64),
        artType=pl.col('artType').map_elements(lambda x: artType_dummy[x], return_dtype=pl.Int32).cast(pl.Float64),
        pubName=pl.col('pubName').map_elements(lambda x: {'DO1':0, 'DO2':1, 'DO3':2}[x], return_dtype=pl.Int32).cast(pl.Float64),
        preType=pl.col('artType').str.split(' ').list.first().map_elements(lambda x: preType_dummy[x], return_dtype=pl.Int32).cast(pl.Float64)

    )
    .with_columns(
        (cs.float() - cs.float().min()) / (cs.float().max() - cs.float().min())
    )
).select(*[f'vector_{i}' for i in range(10)], 'forca', 'artCategory', 'artType', 'preType', 'pubName').to_pandas().corr())

Parece que não há nenhuma relação interessante.

As relações óbvias são do `vector_6` com os subsequentes, porque quando o desdobramento hierárquico chega até o nível 7, é muito provável que ele continue, pea contagem que fizemos.

Além disso, as relações entre `forca` e `artCategory`, já que `artCategory` contém a `forca`, `artType`, `preType` e `pubName` também era esperado.

A única relação que chama a atenção é a relação de `vector_3` com `forca`.

In [ ]:
(
    df_vetores
    .with_columns(
        pl.col('artClass').list.to_struct(
            fields=[f'vector_{i}' for i in range(12)]
        )
    )
    .unnest('artClass')
    .join(
        df,
        on='id'
    )
    .with_columns(
        (cs.float() - cs.float().min()) / (cs.float().max() - cs.float().min()),
        forca=pl.col('artCategory').str.split('/').list.slice(1,1).list.first().map_elements(lambda x: {'Comando da Aeronáutica':0, 'Comando da Marinha':1, 'Comando do Exército':2}[x], return_dtype=pl.Int32),
        artCategory=pl.col('artCategory').map_elements(lambda x: artCategory_dummy[x], return_dtype=pl.Int32),
        pubName=pl.col('pubName').map_elements(lambda x: {'DO1':0, 'DO2':1, 'DO3':2}[x], return_dtype=pl.Int32)

    )
).group_by('vector_3', 'forca').agg(pl.len()).sort('vector_3')

Não existe uma relação tão significativa, até porque são muito valores diferentes de `vector_3` para apenas 3 `forca`.

Vou tentar mais um vez encontrar esse relacionamento, verificando se esses dados de `artClass` podem ser clusterizados de alguma forma.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X = np.vstack(df_vetores.select('artClass').to_series().to_numpy())

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2))
])
X_reduced = pipeline.fit_transform(X)

df_vetores = df_vetores.with_columns(
    pl.Series('vetor_reduzido', X_reduced)
)

In [ ]:
df = df.with_columns(
    vetor=df_vetores.select('vetor_reduzido').to_series()
)

In [ ]:
df.head()

In [ ]:
# px.scatter(
#     df.with_columns(
#         x=pl.col('vetor').cast(pl.List).list.first(),
#         y=pl.col('vetor').cast(pl.List).list.last()
#     ).to_pandas(),
#     x='x',
#     y='y',
#     color='artType',
#     hover_name='id'
# )

sns.scatterplot(
    data = df.with_columns(
         x=pl.col('vetor').cast(pl.List).list.first(),
         y=pl.col('vetor').cast(pl.List).list.last()
     ).to_pandas(),
    x='x',
    y='y',
    hue='artType'
);

Os dados de `artClass` realmente não expressam muito.

Vamos voltar atrás e removê-los do dataframe.

In [ ]:
# df = df.drop('artClass', 'vetor')

# df.head()

### 2.4.3. Valores duplicados

Verificação para existência de dados duplicados.

In [ ]:
df.filter(df.is_duplicated()).shape[0]

Não há dados duplicados, mas só pra garantir:

In [ ]:
df = df.unique()

## 2.5 Visão Geral da Base de Dados

In [ ]:
df.shape

In [ ]:
df.describe()

# 3. Metodologia

In [ ]:
dou2 = df.filter(pl.col('pubName')=='DO2')

In [ ]:
dou2.filter(pl.col('texto').str.to_lowercase().str.contains(r'(decide|resolve):?')).group_by('artType').agg(pl.len()).show(fmt_str_lengths=1500)

In [ ]:
dou2.filter(~pl.col('texto').str.to_lowercase().str.contains(r'(decide|resolve):?')).group_by('artType').agg(pl.len()).show(fmt_str_lengths=1500)

In [ ]:
dou2.filter((~pl.col('texto').str.to_lowercase().str.contains(r'(decide|resolve):?')) & (pl.col('artType')=='Portaria')).show(fmt_str_lengths=1500)

In [ ]:
!spacy download pt_core_news_lg

import nltk
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')

from nltk.corpus import stopwords
pt_stopwords = stopwords.words('portuguese')

nlp = spacy.load('pt_core_news_lg')

In [ ]:
dou2 = (
    dou2
    .with_columns(
        verbos=pl.col('texto').str.to_lowercase().str.extract_all(r'\b[a-zA-Z]+[aeioô]r\b').list.unique()
    )
)

verb_corpus = dou2.select('verbos').to_series().to_list()

verb_set = set(np.hstack(dou2.select('verbos').to_series()).tolist())
verb_set = [i for i in verb_set if nlp(i)[0].pos_ == 'VERB']

verb_corpus = [[i for i in j if i in verb_set] for j in verb_corpus]

verb_corpus = [' '.join(i) for i in verb_corpus]

In [ ]:
vectorizer = TfidfVectorizer(stop_words=pt_stopwords)

tfidf_matrix = vectorizer.fit_transform(verb_corpus)

In [ ]:
pl.DataFrame(
    tfidf_matrix.toarray(),
    schema=vectorizer.get_feature_names_out().tolist(),
    orient='row'
)

In [ ]:
print("Vocabulary without stopwords:", vectorizer.get_feature_names_out())
print("Matrix shape:", tfidf_matrix.shape)

In [ ]:
lista_verbos = list(map(str.lower, np.hstack(dou2.with_columns(
    verbos=pl.col('texto').str.extract_all(r'\b[a-zA-Z]+[aeioô]r\b')
).select('verbos').to_series().to_numpy())))

In [ ]:
dict_verbos = {i:lista_verbos.count(i) for i in set(lista_verbos)}

In [ ]:
(pl.from_dict(dict_verbos)
.drop('por', 'partir', 'militar', 'contar', 'prestador', 'vigor', 'ter', 'ser', 'complementar', 'auxiliar', 'maior', 'diretor', 'parecer', 'valor', 'major', 'salvador', 'instrutor', 'coordenador', 'aviador', 'carater', 'professor', 'servidor', 'assessor')
.transpose(include_header=True).sort('column_0', descending=True))

In [ ]:
(
    dou2
    .with_columns(
        nomeacao=(pl.col('texto').str.to_lowercase().str.contains(r'nomear')),
        exoneracao=(pl.col('texto').str.to_lowercase().str.contains('exonerar')),
        concessao=(pl.col('texto').str.to_lowercase().str.contains('conceder')),
        promocao=(pl.col('texto').str.to_lowercase().str.contains('promover')),
        prorrogacao=(pl.col('texto').str.to_lowercase().str.contains('prorrogar')),
        colocacao=(pl.col('texto').str.to_lowercase().str.contains(r'colocar(.+)disposição')),
        designacao=(pl.col('texto').str.to_lowercase().str.contains('designar')),
        manutencao=(pl.col('texto').str.to_lowercase().str.contains('manter')),
        aposentadoria=(pl.col('texto').str.to_lowercase().str.contains('aposentar')),
        demissao=(pl.col('texto').str.to_lowercase().str.contains('demitir')),
        contratacao=(pl.col('texto').str.to_lowercase().str.contains('contratar')),
        reversao=(pl.col('texto').str.to_lowercase().str.contains('reverter')),
        alteracao=(pl.col('texto').str.to_lowercase().str.contains('alterar')),
        transferencia=(pl.col('texto').str.to_lowercase().str.contains('transferir')),
        renovacao=(pl.col('texto').str.to_lowercase().str.contains(r'renovar')),
        reforma=(pl.col('texto').str.to_lowercase().str.contains(r'reformar')),
        garantia=(pl.col('texto').str.to_lowercase().str.contains(r'assegurar')),
        dispensa=(pl.col('texto').str.to_lowercase().str.contains(r'dispensar')),
        cancelamento=(pl.col('texto').str.to_lowercase().str.contains(r'cancelar')),
        sem_efeito=(pl.col('texto').str.to_lowercase().str.contains(r'tornar sem efeito')),
        anulacao=(pl.col('texto').str.to_lowercase().str.contains(r'anular')),
        consideracao=(pl.col('texto').str.to_lowercase().str.contains(r'considerar')),
        agregacao=(pl.col('texto').str.to_lowercase().str.contains(r'agregar')),
        licenciamento=(pl.col('texto').str.to_lowercase().str.contains(r'licenciar')),
        inclusao=(pl.col('texto').str.to_lowercase().str.contains(r'incluir')),
        incorporacao=(pl.col('texto').str.to_lowercase().str.contains(r'incorporar')),
        declaracao=(pl.col('texto').str.to_lowercase().str.contains(r'declarar')),
        revogacao=(pl.col('texto').str.to_lowercase().str.contains(r'revogar')),
        exclusao=(pl.col('texto').str.to_lowercase().str.contains(r'excluir')),
        autorizacao=(pl.col('texto').str.to_lowercase().str.contains(r'autorizar')),
        convocacao=(pl.col('texto').str.to_lowercase().str.contains(r'convocar')),
        tornar_insub=(pl.col('texto').str.to_lowercase().str.contains(r'tornar insubsistente')),
        suspencao=(pl.col('texto').str.to_lowercase().str.contains(r'suspender')),
        reestabelecimento=(pl.col('texto').str.to_lowercase().str.contains(r'ree?stabelecer')),
        cassacao=(pl.col('texto').str.to_lowercase().str.contains(r'cassar')),
        dispor=(pl.col('texto').str.to_lowercase().str.contains(r'passar [àa] disposição')),
    )
).filter(
    ~pl.any_horizontal(cs.boolean()) & ~pl.col('texto').str.to_lowercase().str.contains('onde se lê')).select('texto').show(fmt_str_lengths=1500)

## 3.1. Identificação das publicações por Força

## 3.2. Padronização

## 3.3. Classificação temática

## 3.4. Critérios de análise

# 4. Análise Exploratória Inicial

## 4.1. Volume total

## 4.2. Distribuição temporal

## 4.3. Distribuição por Força

# 5. Análise Temática

## 5.1. Categorias

## 5.2. Frequência por categoria

## 5.3. Comparação entre Forças

## 5.4. Evolução temporal

# 6. Indicadores e Perfis Institucionais

## 6.1. Indicadores por dimensão

## 6.2. Interpretação comparativa

## 6.3. Síntese dos perfis

# 7. Conclusão

## 7.1. Principais achados

## 7.2. Limitações

## 7.3. Trabalhos futuros